# LeetCode #57 — Insert Interval: chèn khoảng trong một lần duyệt (ba giai đoạn)

**Mức độ:** Medium · **Chủ đề:** Array

Skeleton trên LeetCode:

```python
class Solution:
    def insert(self, intervals: List[List[int]], newInterval: List[int]) -> List[List[int]]:
```

Bài này là "người anh em" của #56 Merge Intervals, với một điểm khác quan trọng: danh sách ban đầu **đã sắp xếp và không chồng nhau**. Tận dụng điều đó, ta giải trong **`O(n)`** thay vì `O(n log n)`.

Mục tiêu của notebook:

1. Chia các khoảng thành **ba nhóm** so với khoảng mới: **trước hẳn**, **chồng nhau**, **sau hẳn**.
2. Viết một vòng lặp với **ba giai đoạn** tuần tự — mẫu xử lý rất hay dùng.
3. Viết điều kiện so sánh **chính xác** (dấu `<` hay `<=`).

# 1. Bài toán là gì?

Cho `intervals` — danh sách khoảng `[start_i, end_i]` **không chồng nhau**, **sắp xếp tăng dần theo `start_i`** — và một khoảng mới `newInterval = [start, end]`.

Chèn `newInterval` sao cho kết quả vẫn **sắp xếp** và **không chồng nhau** (gộp nếu cần). Trả về danh sách mới (không cần sửa tại chỗ).

**Constraints (theo đề):**

- `0 <= intervals.length <= 10^4`
- `intervals[i].length == 2`, `0 <= start_i <= end_i <= 10^5`
- `intervals` sắp xếp tăng dần theo `start_i`, không chồng nhau.
- `newInterval.length == 2`, `0 <= start <= end <= 10^5`

---

## Ví dụ

```text
intervals = [[1, 3], [6, 9]],                         newInterval = [2, 5]
-> [[1, 5], [6, 9]]

intervals = [[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], newInterval = [4, 8]
-> [[1, 2], [3, 10], [12, 16]]                (gộp [3,5], [6,7], [8,10] với [4,8])
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bài toán khoảng (intervals)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bài toán khoảng (intervals)

### Biểu diễn và điều kiện chồng nhau

Một khoảng `[s, e]` gồm điểm đầu `s` và điểm cuối `e` (`s <= e`). Hai khoảng `[a, b]` và `[c, d]` **chồng nhau** (giao khác rỗng, tính cả chạm đầu mút) khi và chỉ khi:

```text
a <= d  and  c <= b          (cách nhớ: "mỗi khoảng bắt đầu trước khi khoảng kia kết thúc")

không chồng:  [a────b]            [c────d]       b < c
chồng:        [a────────b]
                    [c────────d]                 a <= d và c <= b
```

Phần giao (nếu có) là `[max(a, c), min(b, d)]`; hợp khi chồng là `[min(a, c), max(b, d)]`.

In [1]:
def chong_nhau(x, y):
    return x[0] <= y[1] and y[0] <= x[1]

for x_iv, y_iv in [([1, 3], [2, 6]), ([1, 2], [3, 4]), ([1, 4], [4, 5]), ([2, 3], [1, 10])]:
    giao = [max(x_iv[0], y_iv[0]), min(x_iv[1], y_iv[1])] if chong_nhau(x_iv, y_iv) else None
    print(f"{x_iv} và {y_iv}: chồng? {chong_nhau(x_iv, y_iv)} | giao: {giao}")

[1, 3] và [2, 6]: chồng? True | giao: [2, 3]
[1, 2] và [3, 4]: chồng? False | giao: None
[1, 4] và [4, 5]: chồng? True | giao: [4, 4]
[2, 3] và [1, 10]: chồng? True | giao: [2, 3]


### Sắp xếp theo điểm đầu

Hầu hết bài khoảng bắt đầu bằng **sắp xếp theo điểm đầu**. Sau đó, các khoảng có thể gộp với nhau luôn **liền kề** trong thứ tự: chỉ cần so khoảng hiện tại với khoảng **cuối cùng** đã gộp.

```text
sắp xếp: [1,3] [2,6] [8,10] [15,18]
[1,3] + [2,6] chồng → [1,6];  [8,10] không chồng [1,6] → khoảng mới; …
```

In [2]:
def gop(intervals):
    out = []
    for s, e in sorted(intervals):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)       # gộp vào khoảng cuối
        else:
            out.append([s, e])
    return out

print("gộp:", gop([[8, 10], [1, 3], [15, 18], [2, 6]]))

gộp: [[1, 6], [8, 10], [15, 18]]


### Kỹ thuật quét điểm (sweep line)

Biến mỗi khoảng thành hai **sự kiện**: `+1` tại điểm đầu, `−1` sau điểm cuối; sắp xếp sự kiện rồi quét, cộng dồn → biết tại mỗi thời điểm có bao nhiêu khoảng đang "mở" (ví dụ số phòng họp cần dùng).

In [3]:
def so_phong_can(intervals):
    su_kien = []
    for s, e in intervals:
        su_kien.append((s, 1))
        su_kien.append((e, -1))      # khoảng nửa mở [s, e): kết thúc trước khi cái khác bắt đầu cùng lúc
    dang_mo = best = 0
    for _, d in sorted(su_kien):
        dang_mo += d
        best = max(best, dang_mo)
    return best

print("số phòng tối thiểu cho [[0,30],[5,10],[15,20]]:", so_phong_can([[0, 30], [5, 10], [15, 20]]))

số phòng tối thiểu cho [[0,30],[5,10],[15,20]]: 2


Lỗi hay gặp: dùng `<` thay `<=` (hoặc ngược lại) — đề quy định khoảng **chạm nhau** có tính là chồng không; quên sắp xếp; gộp bằng `e` mới thay vì `max(e_cũ, e_mới)` (khoảng sau có thể nằm gọn trong khoảng trước).

## 2.2 Áp dụng vào bài này

Danh sách **khoảng** đã sắp xếp, không chồng nhau; chèn khoảng mới theo ba giai đoạn: các khoảng kết thúc **trước** khoảng mới; các khoảng **chồng** với nó (gộp bằng `min`/`max`); các khoảng bắt đầu **sau** nó. Điều kiện chồng nhau chính là `a <= d and c <= b`.

# 3. Cách dễ nghĩ: thêm vào rồi gộp như bài #56

In [4]:
from typing import List

def merge(intervals: List[List[int]]) -> List[List[int]]:
    intervals = sorted(intervals, key=lambda iv: iv[0])
    result = []
    for s, e in intervals:
        if result and s <= result[-1][1]:
            result[-1][1] = max(result[-1][1], e)
        else:
            result.append([s, e])
    return result


def insert_via_merge(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    return merge(intervals + [new])

print(insert_via_merge([[1, 3], [6, 9]], [2, 5]))
print(insert_via_merge([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))

[[1, 5], [6, 9]]
[[1, 2], [3, 10], [12, 16]]


**Đúng**, nhưng `O(n log n)` vì phải sắp xếp lại — **bỏ phí** thông tin "danh sách đã sắp xếp sẵn". Ta có thể làm `O(n)`.

# 4. Ý tưởng: ba nhóm

So với `new = [s, e]`, mỗi khoảng `[a, b]` thuộc đúng **một** trong ba nhóm:

| Nhóm | Điều kiện | Xử lý |
|---|---|---|
| **Trước hẳn** | `b < s` (kết thúc trước khi `new` bắt đầu) | giữ nguyên |
| **Chồng nhau** | `a <= e` và `b >= s` | gộp vào `new` |
| **Sau hẳn** | `a > e` (bắt đầu sau khi `new` kết thúc) | giữ nguyên |

Vì danh sách đã sắp xếp và rời nhau, ba nhóm này xuất hiện **liên tiếp theo đúng thứ tự**: mọi khoảng "trước hẳn" đứng đầu, rồi tới các khoảng "chồng nhau", rồi tới các khoảng "sau hẳn".

```text
            [1,2]   [3,5]  [6,7]  [8,10]   [12,16]
new =                  [4 ........ 8]
nhóm:      trước   |  chồng  chồng  chồng |  sau
```

Chú ý: chạm nhau (`b == s` hoặc `a == e`) tính là **chồng** → phải gộp. Vì vậy điều kiện "trước hẳn" dùng **`<`**, "sau hẳn" dùng **`>`**.

In [5]:
intervals = [[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]]
s, e = 4, 8
for a, b in intervals:
    group = "trước hẳn" if b < s else ("sau hẳn" if a > e else "chồng nhau")
    print(f"[{a}, {b}] -> {group}")

[1, 2] -> trước hẳn
[3, 5] -> chồng nhau
[6, 7] -> chồng nhau
[8, 10] -> chồng nhau
[12, 16] -> sau hẳn


# 5. Lời giải chính: một vòng lặp, ba giai đoạn

```python
i, n = 0, len(intervals)

# Giai đoạn 1: các khoảng trước hẳn
while i < n and intervals[i][1] < s:
    result.append(intervals[i]); i += 1

# Giai đoạn 2: gộp mọi khoảng chồng nhau vào new
while i < n and intervals[i][0] <= e:
    s = min(s, intervals[i][0])
    e = max(e, intervals[i][1])
    i += 1
result.append([s, e])

# Giai đoạn 3: các khoảng sau hẳn
result.extend(intervals[i:])
```

Ở giai đoạn 2 chỉ cần kiểm tra `intervals[i][0] <= e`, vì sau giai đoạn 1 ta đã biết chắc `intervals[i][1] >= s`.

In [6]:
def insert(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    result = []
    s, e = new
    i, n = 0, len(intervals)

    while i < n and intervals[i][1] < s:          # 1. trước hẳn
        result.append(intervals[i])
        i += 1

    while i < n and intervals[i][0] <= e:         # 2. chồng nhau -> gộp
        s = min(s, intervals[i][0])
        e = max(e, intervals[i][1])
        i += 1
    result.append([s, e])

    result.extend(intervals[i:])                  # 3. sau hẳn
    return result

print(insert([[1, 3], [6, 9]], [2, 5]))
print(insert([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))
print(insert([], [5, 7]))

[[1, 5], [6, 9]]
[[1, 2], [3, 10], [12, 16]]
[[5, 7]]


# 6. Mô phỏng từng bước

In [7]:
def trace_insert(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    result = []
    s, e = new
    i, n = 0, len(intervals)
    print(f"intervals = {intervals}, new = {new}")
    while i < n and intervals[i][1] < s:
        print(f"  GĐ1: {intervals[i]} kết thúc {intervals[i][1]} < {s} -> giữ nguyên")
        result.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= e:
        old = [s, e]
        s = min(s, intervals[i][0])
        e = max(e, intervals[i][1])
        print(f"  GĐ2: {intervals[i]} bắt đầu {intervals[i][0]} <= {old[1]} -> gộp: {old} thành [{s}, {e}]")
        i += 1
    result.append([s, e])
    print(f"  Thêm khoảng đã gộp [{s}, {e}]")
    if i < n:
        print(f"  GĐ3: giữ nguyên phần còn lại {intervals[i:]}")
    result.extend(intervals[i:])
    print("  Kết quả:", result)
    return result

_ = trace_insert([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8])
print()
_ = trace_insert([[1, 5]], [6, 8])
print()
_ = trace_insert([[3, 5]], [1, 2])

intervals = [[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], new = [4, 8]
  GĐ1: [1, 2] kết thúc 2 < 4 -> giữ nguyên
  GĐ2: [3, 5] bắt đầu 3 <= 8 -> gộp: [4, 8] thành [3, 8]
  GĐ2: [6, 7] bắt đầu 6 <= 8 -> gộp: [3, 8] thành [3, 8]
  GĐ2: [8, 10] bắt đầu 8 <= 8 -> gộp: [3, 8] thành [3, 10]
  Thêm khoảng đã gộp [3, 10]
  GĐ3: giữ nguyên phần còn lại [[12, 16]]
  Kết quả: [[1, 2], [3, 10], [12, 16]]

intervals = [[1, 5]], new = [6, 8]
  GĐ1: [1, 5] kết thúc 5 < 6 -> giữ nguyên
  Thêm khoảng đã gộp [6, 8]
  Kết quả: [[1, 5], [6, 8]]

intervals = [[3, 5]], new = [1, 2]
  Thêm khoảng đã gộp [1, 2]
  GĐ3: giữ nguyên phần còn lại [[3, 5]]
  Kết quả: [[1, 2], [3, 5]]


Hai ví dụ cuối cho thấy khoảng mới có thể nằm **sau tất cả** (giai đoạn 2 và 3 rỗng) hoặc **trước tất cả** (giai đoạn 1 và 2 rỗng).

# 7. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def insert(self, intervals: List[List[int]], newInterval: List[int]) -> List[List[int]]:
        result = []
        s, e = newInterval
        i, n = 0, len(intervals)
        while i < n and intervals[i][1] < s:
            result.append(intervals[i])
            i += 1
        while i < n and intervals[i][0] <= e:
            s = min(s, intervals[i][0])
            e = max(e, intervals[i][1])
            i += 1
        result.append([s, e])
        result.extend(intervals[i:])
        return result

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.insert([[1, 3], [6, 9]], [2, 5]), [[1, 5], [6, 9]])
check("Ví dụ 2", sol.insert([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]), [[1, 2], [3, 10], [12, 16]])
check("Danh sách rỗng", sol.insert([], [5, 7]), [[5, 7]])
check("Chèn vào đầu", sol.insert([[3, 5]], [1, 2]), [[1, 2], [3, 5]])
check("Chèn vào cuối", sol.insert([[1, 5]], [6, 8]), [[1, 5], [6, 8]])
check("Chạm bên trái", sol.insert([[1, 5]], [0, 1]), [[0, 5]])
check("Chạm bên phải", sol.insert([[1, 5]], [5, 7]), [[1, 7]])
check("Nằm trọn bên trong", sol.insert([[1, 10]], [3, 4]), [[1, 10]])
check("Bao trùm tất cả", sol.insert([[2, 3], [5, 6]], [1, 9]), [[1, 9]])
check("Lọt vào khe", sol.insert([[1, 2], [6, 7]], [3, 4]), [[1, 2], [3, 4], [6, 7]])

import random
random.seed(57)
for _ in range(3000):
    base = merge([[a, a + random.randint(0, 4)] for a in random.sample(range(0, 40), random.randint(0, 8))])
    # merge có thể làm các khoảng chạm nhau dính lại -> base hợp lệ (rời nhau, sắp xếp)
    a = random.randint(0, 45)
    new = [a, a + random.randint(0, 8)]
    assert sol.insert([iv[:] for iv in base], new) == insert_via_merge(base, new), (base, new)
print("3000 test ngẫu nhiên: khớp cách thêm-rồi-gộp")

Ví dụ 1                      PASS  [[1, 5], [6, 9]]
Ví dụ 2                      PASS  [[1, 2], [3, 10], [12, 16]]
Danh sách rỗng               PASS  [[5, 7]]
Chèn vào đầu                 PASS  [[1, 2], [3, 5]]
Chèn vào cuối                PASS  [[1, 5], [6, 8]]
Chạm bên trái                PASS  [[0, 5]]
Chạm bên phải                PASS  [[1, 7]]
Nằm trọn bên trong           PASS  [[1, 10]]
Bao trùm tất cả              PASS  [[1, 9]]
Lọt vào khe                  PASS  [[1, 2], [3, 4], [6, 7]]


3000 test ngẫu nhiên: khớp cách thêm-rồi-gộp


# 8. Vì sao đúng? Độ phức tạp?

**Đúng:**

- **Giai đoạn 1** dừng ở khoảng đầu tiên có `end >= s`. Mọi khoảng trước nó kết thúc trước `s` → không chạm `new`, giữ nguyên.
- **Giai đoạn 2**: mỗi khoảng có `start <= e` (và `end >= s` từ giai đoạn 1) chồng với `[s, e]` hiện tại → gộp, mở rộng `[s, e]`. Dừng ở khoảng đầu tiên có `start > e` (với `e` **đã mở rộng**).
- **Giai đoạn 3**: mọi khoảng còn lại bắt đầu sau `e` (vì sắp xếp) → không chạm, giữ nguyên.

Thứ tự kết quả tự nhiên đúng vì ba nhóm nối tiếp nhau.

**Độ phức tạp:**

```text
Time:  O(n)  — mỗi khoảng xét đúng một lần
Space: O(n)  — danh sách kết quả
```

**Có thể dùng Binary Search?** Có thể tìm vị trí giai đoạn 1 / 3 trong `O(log n)`, nhưng việc tạo danh sách kết quả vẫn tốn `O(n)`, nên không cải thiện độ phức tạp tổng.

# 9. Những lỗi dễ mắc

## Lỗi 1: Dùng `<=` ở giai đoạn 1 (bỏ sót khoảng chạm)

In [10]:
def wrong_phase1(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    result = []
    s, e = new
    i, n = 0, len(intervals)
    while i < n and intervals[i][1] <= s:            # SAI: chạm nhau phải gộp
        result.append(intervals[i]); i += 1
    while i < n and intervals[i][0] <= e:
        s = min(s, intervals[i][0]); e = max(e, intervals[i][1]); i += 1
    result.append([s, e])
    result.extend(intervals[i:])
    return result

print("Sai :", wrong_phase1([[1, 5]], [5, 7]), "| Đúng:", sol.insert([[1, 5]], [5, 7]))

Sai : [[1, 5], [5, 7]] | Đúng: [[1, 7]]


## Lỗi 2: Dùng `<` ở giai đoạn 2

Khoảng bắt đầu đúng tại `e` (chạm bên phải) sẽ không được gộp.

In [11]:
def wrong_phase2(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    result = []
    s, e = new
    i, n = 0, len(intervals)
    while i < n and intervals[i][1] < s:
        result.append(intervals[i]); i += 1
    while i < n and intervals[i][0] < e:             # SAI
        s = min(s, intervals[i][0]); e = max(e, intervals[i][1]); i += 1
    result.append([s, e])
    result.extend(intervals[i:])
    return result

print("Sai :", wrong_phase2([[5, 9]], [1, 5]), "| Đúng:", sol.insert([[5, 9]], [1, 5]))

Sai : [[1, 5], [5, 9]] | Đúng: [[1, 9]]


## Lỗi 3: Quên `min` cho điểm đầu

Khoảng chồng đầu tiên có thể bắt đầu **trước** `s` (ví dụ `[1, 3]` với `new = [2, 5]`). Phải `s = min(s, start)`.

## Lỗi 4: Quên thêm khoảng đã gộp khi danh sách rỗng hoặc khi mọi khoảng đều "trước hẳn"

Dòng `result.append([s, e])` phải nằm **ngoài** vòng `while` của giai đoạn 2, để luôn được chạy.

# 10. Mẹo nhận diện pattern

**Dữ liệu đã sắp xếp → một lần duyệt chia giai đoạn:**

```python
i = 0
while i < n and <thuộc nhóm 1>: xử lý; i += 1
while i < n and <thuộc nhóm 2>: xử lý; i += 1
phần còn lại là nhóm 3
```

Mẫu này xuất hiện ở: **#57 Insert Interval**, **#88 Merge Sorted Array**, **#986 Interval List Intersections**, **#1272 Remove Interval**.

Ghi nhớ quy ước so sánh với khoảng **đóng** `[a, b]`:

- Rời nhau: `b < c` hoặc `d < a`.
- Chồng / chạm: `a <= d and c <= b`.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chèn `[0, 20]` vào `[[1, 2], [5, 6], [19, 25]]`, và chèn `[7, 7]` vào `[[1, 2], [5, 6], [8, 9]]`. Ghi rõ từng giai đoạn.

## Bài 2 — Điền chỗ trống

In [12]:
def exercise_2_insert(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    result = []
    s, e = new
    i, n = 0, len(intervals)
    # TODO 1: giai đoạn 1 — các khoảng kết thúc trước s
    # TODO 2: giai đoạn 2 — gộp các khoảng bắt đầu không sau e
    # TODO 3: thêm [s, e], rồi phần còn lại
    return result


# assert exercise_2_insert([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
# assert exercise_2_insert([], [5, 7]) == [[5, 7]]
# assert exercise_2_insert([[1, 5]], [5, 7]) == [[1, 7]]

## Bài 3 — Biến thể: Remove Interval (LeetCode #1272)

Các khoảng ở đây là **nửa mở** `[a, b)` (gồm `a`, không gồm `b`), đã sắp xếp và rời nhau. **Xóa** phần giao với `toBeRemoved = [x, y)`.

```python
remove_interval([[0, 2], [3, 4], [5, 7]], [1, 6]) -> [[0, 1], [6, 7]]
remove_interval([[0, 5]], [2, 3])                 -> [[0, 2], [3, 5]]
remove_interval([[-5, -4], [-3, -2], [1, 2], [3, 5], [8, 9]], [-1, 4]) -> [[-5, -4], [-3, -2], [4, 5], [8, 9]]
```

In [13]:
def exercise_3_remove_interval(intervals: List[List[int]], to_remove: List[int]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_3_remove_interval([[0, 2], [3, 4], [5, 7]], [1, 6]) == [[0, 1], [6, 7]]
# assert exercise_3_remove_interval([[0, 5]], [2, 3]) == [[0, 2], [3, 5]]
# assert exercise_3_remove_interval([[-5, -4], [-3, -2], [1, 2], [3, 5], [8, 9]], [-1, 4]) == [[-5, -4], [-3, -2], [4, 5], [8, 9]]

## Bài 4 — Biến thể: giao của hai danh sách khoảng (LeetCode #986)

Hai danh sách khoảng **đóng**, mỗi danh sách đã sắp xếp và rời nhau. Trả về giao của chúng.

```python
intersect([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]])
-> [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]
```

Gợi ý nhanh: hai con trỏ; giao của `[a, b]` và `[c, d]` là `[max(a, c), min(b, d)]` nếu hợp lệ; dời con trỏ của khoảng **kết thúc sớm hơn**.

In [14]:
def exercise_4_intersect(A: List[List[int]], B: List[List[int]]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_4_intersect([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]]) == [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]
# assert exercise_4_intersect([[1, 3]], []) == []

## Bài 5 — Mini challenge: chèn nhiều khoảng

Chèn **lần lượt** một danh sách `news` (không sắp xếp) vào `intervals`. So sánh hai cách:

1. Gọi `insert` cho từng khoảng: `O(k · n)`.
2. Gộp tất cả rồi `merge` một lần: `O((n + k) log(n + k))`.

Viết cách 1 và kiểm tra nó cho cùng kết quả với cách 2.

```python
insert_many([[1, 2], [8, 9]], [[3, 4], [2, 3], [10, 12]]) -> [[1, 4], [8, 9], [10, 12]]
```

In [15]:
def exercise_5_insert_many(intervals: List[List[int]], news: List[List[int]]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_5_insert_many([[1, 2], [8, 9]], [[3, 4], [2, 3], [10, 12]]) == [[1, 4], [8, 9], [10, 12]]
# assert exercise_5_insert_many([], [[5, 6], [1, 2]]) == [[1, 2], [5, 6]]

# 12. Gợi ý

**Bài 1**
- Mức 1: `[0, 20]` chồng với cả ba khoảng; `[7, 7]` lọt vào khe giữa `[5, 6]` và `[8, 9]`.

**Bài 2**
- Mức 1: so với code mục 5.

**Bài 3**
- Mức 1: với mỗi `[a, b)`: nếu không giao (`b <= x` hoặc `a >= y`) → giữ nguyên.
- Mức 2: nếu giao: giữ phần trái `[a, x)` khi `a < x`, và phần phải `[y, b)` khi `y < b`.

**Bài 4**
- Mức 1: `lo = max(a, c)`, `hi = min(b, d)`; nếu `lo <= hi` thì thêm `[lo, hi]`.
- Mức 2: `if b < d: i += 1 else: j += 1`.

**Bài 5**
- Mức 1: `cur = intervals`; với mỗi `new` trong `news`: `cur = insert(cur, new)`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [16]:
_ = trace_insert([[1, 2], [5, 6], [19, 25]], [0, 20])
print()
_ = trace_insert([[1, 2], [5, 6], [8, 9]], [7, 7])

intervals = [[1, 2], [5, 6], [19, 25]], new = [0, 20]
  GĐ2: [1, 2] bắt đầu 1 <= 20 -> gộp: [0, 20] thành [0, 20]
  GĐ2: [5, 6] bắt đầu 5 <= 20 -> gộp: [0, 20] thành [0, 20]
  GĐ2: [19, 25] bắt đầu 19 <= 20 -> gộp: [0, 20] thành [0, 25]
  Thêm khoảng đã gộp [0, 25]
  Kết quả: [[0, 25]]

intervals = [[1, 2], [5, 6], [8, 9]], new = [7, 7]
  GĐ1: [1, 2] kết thúc 2 < 7 -> giữ nguyên
  GĐ1: [5, 6] kết thúc 6 < 7 -> giữ nguyên
  Thêm khoảng đã gộp [7, 7]
  GĐ3: giữ nguyên phần còn lại [[8, 9]]
  Kết quả: [[1, 2], [5, 6], [7, 7], [8, 9]]


## Đáp án Bài 2

In [17]:
def answer_2_insert(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    result = []
    s, e = new
    i, n = 0, len(intervals)
    while i < n and intervals[i][1] < s:
        result.append(intervals[i]); i += 1
    while i < n and intervals[i][0] <= e:
        s = min(s, intervals[i][0]); e = max(e, intervals[i][1]); i += 1
    result.append([s, e])
    result.extend(intervals[i:])
    return result

assert answer_2_insert([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert answer_2_insert([], [5, 7]) == [[5, 7]]
assert answer_2_insert([[1, 5]], [5, 7]) == [[1, 7]]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [18]:
def answer_3_remove_interval(intervals: List[List[int]], to_remove: List[int]) -> List[List[int]]:
    x, y = to_remove
    result = []
    for a, b in intervals:
        if b <= x or a >= y:              # không giao (khoảng nửa mở)
            result.append([a, b])
        else:
            if a < x:
                result.append([a, x])
            if y < b:
                result.append([y, b])
    return result

assert answer_3_remove_interval([[0, 2], [3, 4], [5, 7]], [1, 6]) == [[0, 1], [6, 7]]
assert answer_3_remove_interval([[0, 5]], [2, 3]) == [[0, 2], [3, 5]]
assert answer_3_remove_interval([[-5, -4], [-3, -2], [1, 2], [3, 5], [8, 9]], [-1, 4]) == [[-5, -4], [-3, -2], [4, 5], [8, 9]]
print("Bài 3 PASS")

Bài 3 PASS


Chú ý: với khoảng **nửa mở** `[a, b)`, "không giao" là `b <= x` (có dấu `=`), khác với khoảng đóng ở bài chính. Luôn đọc kỹ đề dùng loại khoảng nào.

## Đáp án Bài 4

In [19]:
def answer_4_intersect(A: List[List[int]], B: List[List[int]]) -> List[List[int]]:
    i = j = 0
    result = []
    while i < len(A) and j < len(B):
        lo = max(A[i][0], B[j][0])
        hi = min(A[i][1], B[j][1])
        if lo <= hi:
            result.append([lo, hi])
        if A[i][1] < B[j][1]:
            i += 1
        else:
            j += 1
    return result

assert answer_4_intersect([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]]) == [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]
assert answer_4_intersect([[1, 3]], []) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [20]:
def answer_5_insert_many(intervals: List[List[int]], news: List[List[int]]) -> List[List[int]]:
    cur = [iv[:] for iv in intervals]
    for new in news:
        cur = answer_2_insert(cur, new)
    return cur

assert answer_5_insert_many([[1, 2], [8, 9]], [[3, 4], [2, 3], [10, 12]]) == [[1, 4], [8, 9], [10, 12]]
assert answer_5_insert_many([], [[5, 6], [1, 2]]) == [[1, 2], [5, 6]]
for _ in range(1000):
    base = merge([[a, a + random.randint(0, 3)] for a in random.sample(range(0, 30), random.randint(0, 5))])
    news = [[a, a + random.randint(0, 5)] for a in (random.randint(0, 35) for _ in range(random.randint(1, 4)))]
    assert answer_5_insert_many(base, news) == merge(base + news)
print("Bài 5 PASS (chèn từng khoảng khớp gộp một lần trên 1000 test)")

Bài 5 PASS (chèn từng khoảng khớp gộp một lần trên 1000 test)

# 14. Tổng kết

> Danh sách đã sắp xếp và rời nhau → các khoảng chia thành **ba nhóm liên tiếp**: trước hẳn (`end < s`), chồng nhau (`start <= e`), sau hẳn. Một vòng duyệt với ba giai đoạn cho lời giải `O(n)`.

---

## Checklist trước khi nộp

- [ ] Giai đoạn 1 dùng `intervals[i][1] < s` (dấu `<`)?
- [ ] Giai đoạn 2 dùng `intervals[i][0] <= e` (dấu `<=`), cập nhật cả `min` và `max`?
- [ ] Luôn thêm `[s, e]` sau giai đoạn 2?
- [ ] Đã thử danh sách rỗng, chèn đầu, chèn cuối, chạm hai bên, bao trùm tất cả?